# 7CS537 CW2, Notebook 03: Data verification and time-ordered splits

**Student number:** 100810453

**Purpose:** before any model is trained for CW2, check that the cleaned CICIDS2017 data is exactly the data used in CW1, confirm which attacks happen on which capture day, and build the train, validation and test splits that Notebooks 04 to 06 will use.

**Why this notebook comes first:** CW1's main limitation was the random split. Flows from the same attack session could land in both training and test data, so test scores were probably optimistic. CW2 instead trains on earlier days and tests on later days, as a SOC would in practice. That only works if the day boundaries and labels are verified first, so every later result rests on this notebook.

**Experiments whose splits are built here:**

| Name | Training data | Test data | What it answers |
|---|---|---|---|
| E1 (main) | Monday to Thursday | Friday | How does the system cope when new attack families arrive later in time? |
| E2 (robustness) | Only the days before each test day | Wednesday, then Thursday (Friday is E1) | Does the E1 finding hold on other days, still in time order? |
| Autoencoder | Benign Monday flows only | Any later day | Can a model that has only seen normal traffic flag attacks? |
| Random (reference) | 70% random, as in CW1 | 15% random | How well are attack families the model has already seen detected? Used for the reference result and the held-out attack family experiment (E3) |

**Inputs (from CW1 Notebook 01, run from the repository root):** `data/cicids2017_clean.parquet`, `outputs/01_feature_list.txt`, `outputs/01_run_config.json`, `outputs/02_split_counts.csv`.

**Outputs:** `cw2/data/03_splits.parquet` (split assignment for every flow; not committed, recreated by running this notebook) and evidence files in `cw2/outputs/` with the prefix `03_`.

**How to run:** open this notebook from the `cw2/` folder of the repository and run all cells from a fresh kernel. It does not change any CW1 file.

## 1. Setup and configuration
**What:** imports the libraries and sets every option in one place.

**Why:**
- `RANDOM_STATE = 42` is the same seed as CW1, so the random reference split is identical to CW1's and a re-run gives the same splits.
- `VAL_FRAC` keeps 15% of each training period back for validation. Validation data is used for early stopping and for choosing alert thresholds, never the test day.
- `AE_VAL_FRAC` keeps 20% of the benign Monday flows back so that autoencoder training can be stopped before it over-fits.
- Paths are relative, so the notebook works on any computer where the repository has been cloned and Notebook 01 has been run.

In [ ]:
import os
import json
import hashlib
import platform

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42     # same seed as CW1
VAL_FRAC = 0.15       # share of each training period kept back for validation
AE_VAL_FRAC = 0.20    # share of benign Monday flows kept back to stop autoencoder training

# This notebook lives in cw2/. The CW1 data and outputs are one level up (the repository root).
ROOT = ".." if os.path.exists(os.path.join("..", "outputs", "01_feature_list.txt")) else "."
DATA_FILE = os.path.join(ROOT, "data", "cicids2017_clean.parquet")
CW1_OUT = os.path.join(ROOT, "outputs")
CW2_DIR = os.path.join(ROOT, "cw2")
OUT_DIR = os.path.join(CW2_DIR, "outputs")
SPLIT_FILE = os.path.join(CW2_DIR, "data", "03_splits.parquet")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SPLIT_FILE), exist_ok=True)

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

print("Repository root:", os.path.abspath(ROOT))
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| numpy", np.__version__)

## 2. Load the cleaned data and check it is the CW1 data
**What:** loads the cleaned file from Notebook 01 and the list of 62 features, then checks the row count, feature count, column names and file names against the configuration CW1 saved.

**Why:** if any of these differ, the CW1 and CW2 results would not be comparable. The `assert` lines stop the notebook with a clear message instead of carrying on with the wrong data.

A short fingerprint of the label and file columns is also saved. Notebooks 04 to 06 recompute it to prove they are reading the same rows in the same order, because the split file refers to rows by position.

In [ ]:
if os.path.exists(DATA_FILE):
    df = pd.read_parquet(DATA_FILE)
else:
    df = pd.read_pickle(DATA_FILE.replace(".parquet", ".pkl"))

with open(os.path.join(CW1_OUT, "01_feature_list.txt")) as fh:
    feature_cols = [line.strip() for line in fh if line.strip()]
with open(os.path.join(CW1_OUT, "01_run_config.json")) as fh:
    cw1_config = json.load(fh)

# Plain strings make grouping and comparison simpler (as in CW1 Notebook 02)
for col in ["Label", "attack_family", "source_file"]:
    df[col] = df[col].astype(str)

needed = feature_cols + ["Label", "attack_family", "is_attack", "source_file"]
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Columns missing from the cleaned data: {missing}"
assert len(feature_cols) == cw1_config["n_features"], "Feature list does not match CW1"
assert len(df) == cw1_config["rows_final"], f"Row count {len(df):,} differs from CW1 ({cw1_config['rows_final']:,})"
assert set(df["source_file"]) == set(cw1_config["files"]), "Capture files differ from CW1"
assert (df["is_attack"] == (df["Label"] != "BENIGN")).all(), "is_attack does not match Label"

def data_fingerprint(frame):
    # Hash of the label and file of every row, in order. Same data in the same order gives the same value.
    h = pd.util.hash_pandas_object(frame[["Label", "source_file"]], index=False).to_numpy()
    return hashlib.sha256(h.tobytes()).hexdigest()[:16]

FINGERPRINT = data_fingerprint(df)
print(f"{len(df):,} rows, {len(feature_cols)} features, {df['source_file'].nunique()} capture files")
print("Data fingerprint:", FINGERPRINT)

**What this shows** (run of 14 Nov 2026): 2,496,544 rows and 62 features, identical to CW1. The data fingerprint is `cb8a0f0b8bd9cf88`; Notebooks 04 to 06 check it before using the splits.

## 3. Map each capture file to its day
**What:** takes the day from the start of each file name (for example `Friday-WorkingHours-Morning...` gives Friday) and orders the days Monday to Friday.

**Why:** the CSV files have no timestamp column, so the file name is the only record of when a flow was captured. Three files belong to Friday and two to Thursday, so splitting by day, not by file, keeps each day whole.

In [ ]:
df["day"] = df["source_file"].str.split("-").str[0]
unknown = set(df["day"]) - set(DAY_ORDER)
assert not unknown, f"Unexpected day names: {unknown}"
df["day"] = pd.Categorical(df["day"], categories=DAY_ORDER, ordered=True)

file_day = (df.groupby("source_file", observed=True)
              .agg(day=("day", "first"), flows=("Label", "size"), attack_flows=("is_attack", "sum"))
              .sort_values("day"))
file_day.to_csv(f"{OUT_DIR}/03_files_by_day.csv")
display(file_day)

**What this shows** (`03_files_by_day.csv`): the 8 files map to 5 days. Friday is the largest day (614,833 flows from three files) and Thursday has two files (396,284 flows).

## 4. Which attacks happen on which day?
**What:** counts every label and every attack family on each day, then checks three facts the CW2 design depends on:
1. Monday contains no attacks, so it can be used as a period of purely normal traffic.
2. Each attack label appears on one day only.
3. Which Friday attack families never appear from Monday to Thursday.

**Why:** fact 1 justifies training the autoencoder on Monday. Fact 3 is the basis of E1: families that only appear on Friday are genuinely new to a model trained on earlier days. Fact 2 has a consequence the design must state openly: in any time-ordered test, every attack family on the test day is new to XGBoost. So the time-ordered tests cannot measure how well XGBoost detects families it has already seen. That figure comes from the random reference split instead.

In [ ]:
label_by_day = pd.crosstab(df["Label"], df["day"])
family_by_day = pd.crosstab(df["attack_family"], df["day"])
label_by_day.to_csv(f"{OUT_DIR}/03_label_by_day.csv")
family_by_day.to_csv(f"{OUT_DIR}/03_family_by_day.csv")
display(label_by_day)

attacks = label_by_day.drop(index="BENIGN")
assert attacks["Monday"].sum() == 0, "Monday contains attack flows"
days_per_label = attacks.gt(0).sum(axis=1)
print("Number of days each attack label appears on:")
print(days_per_label.to_string())
assert (days_per_label == 1).all(), "Some attack label appears on more than one day"

is_attack = df["is_attack"].to_numpy() == 1
friday = (df["day"] == "Friday").to_numpy()
friday_families = set(df.loc[friday & is_attack, "attack_family"])
earlier_families = set(df.loc[~friday & is_attack, "attack_family"])
print("\nFriday attack families:", sorted(friday_families))
print("Families seen Monday to Thursday:", sorted(earlier_families))
print("Friday families never seen Monday to Thursday:", sorted(friday_families - earlier_families))

**What this shows** (`03_label_by_day.csv`, `03_family_by_day.csv`):
- Monday has 492,576 benign flows and no attacks, so it is a clean period of normal traffic for the autoencoder.
- Every one of the 14 attack labels appears on exactly one day. So in any time-ordered test, all attacks on the test day belong to families the supervised model has never seen. Known-family detection is therefore measured on the random reference split (Section 7), not here.
- Friday's three families, Botnet (1,948 flows), DDoS (128,011) and PortScan (90,130), never appear from Monday to Thursday. E1 is therefore a genuine test of new attack families arriving later in time.
- 35.8% of Friday's flows are attacks (220,089 of 614,833), far more than on a real network. Precision on Friday will look better than it would in TeleComNet's real traffic, where attacks are rare, so false alarms per day are reported as well.

## 5. Build the time-ordered splits (E1 and E2)
**What:** for a chosen test day, all earlier days form the training period. 15% of that period is kept back as validation, stratified by the original label so that rare attacks such as Heartbleed appear in both parts. The whole test day is the test set.

- **E1:** test day Friday, training Monday to Thursday.
- **E2:** test day Wednesday (training Monday and Tuesday) and test day Thursday (training Monday to Wednesday). E2 for Friday is identical to E1, so it is not stored twice.
- **Tuesday cannot be a supervised test day:** the only earlier day is Monday, which has no attacks, so XGBoost would have nothing to learn from. Tuesday can still be used to test the autoencoder.

**Why a random validation slice inside the training period:** validation is only used to stop training and choose thresholds, so it must come from the training period. Taking it from the test day would let test information leak into those choices.

In [ ]:
idx_all = np.arange(len(df))
labels = df["Label"].to_numpy()
SPLIT_CATS = ["train", "val", "test", "unused"]

def chronological_split(test_day):
    # Train and validation from every day before test_day; test = the whole test_day
    train_days = DAY_ORDER[:DAY_ORDER.index(test_day)]
    pool = idx_all[df["day"].isin(train_days).to_numpy()]
    tr, va = train_test_split(pool, test_size=VAL_FRAC, stratify=labels[pool], random_state=RANDOM_STATE)
    te = idx_all[(df["day"] == test_day).to_numpy()]
    role = np.full(len(df), "unused", dtype=object)
    role[tr], role[va], role[te] = "train", "val", "test"
    return pd.Categorical(role, categories=SPLIT_CATS)

splits = pd.DataFrame({
    "row_id": idx_all,
    "day": df["day"].to_numpy(),
    "Label": labels,
    "attack_family": df["attack_family"].to_numpy(),
    "is_attack": df["is_attack"].to_numpy(),
})
splits["E1"] = chronological_split("Friday")
splits["E2_Wednesday"] = chronological_split("Wednesday")
splits["E2_Thursday"] = chronological_split("Thursday")

for col in ["E1", "E2_Wednesday", "E2_Thursday"]:
    print(col, splits[col].value_counts().reindex(SPLIT_CATS).to_dict())

**What this shows** (`03_split_counts_E1.csv`, `03_split_counts_E2_*.csv`):
- **E1:** 1,599,454 training, 282,257 validation and 614,833 test flows. No Bot, DDoS or PortScan flow is in training or validation. The rarest training labels are small: Heartbleed 9, SQL injection 18 and Infiltration 31 training flows.
- **E2, Wednesday:** the only attacks in training are 7,777 brute-force flows (FTP and SSH). The test day has 193,590 DoS and Heartbleed flows, so this asks whether knowing brute force helps to spot DoS.
- **E2, Thursday:** only 2,179 of the 396,284 test flows are attacks (0.55%), and Infiltration (36) and SQL injection (21) are tiny. Per-label rates on Thursday will be unreliable for those two labels, and precision will be very sensitive to false alarms.

## 6. Autoencoder data (benign Monday only)
**What:** takes every benign Monday flow and divides it at random into 80% for training the autoencoder and 20% for stopping its training early.

**Why:** an anomaly detector must learn what normal traffic looks like without ever seeing an attack, so its training data has to be purely benign. Monday is the only day with no attacks at all.

**How the alert threshold will be set (Notebook 05):** the threshold is chosen on benign traffic only, from a period separate from training, and before the test day. For test day D, the calibration set is the 20% Monday hold-out plus the benign flows of every day from Tuesday up to the day before D. For Friday (E1) that is the Monday hold-out plus benign Tuesday to Thursday flows. Attack labels and the test day are never used to choose it. The helper below builds that set so Notebook 05 cannot get it wrong.

In [ ]:
monday_benign = idx_all[((df["day"] == "Monday") & (df["is_attack"] == 0)).to_numpy()]
ae_train, ae_val = train_test_split(monday_benign, test_size=AE_VAL_FRAC, random_state=RANDOM_STATE)
ae_role = np.full(len(df), "not_used_for_training", dtype=object)
ae_role[ae_train], ae_role[ae_val] = "ae_train", "ae_val"
splits["ae_role"] = pd.Categorical(ae_role, categories=["ae_train", "ae_val", "not_used_for_training"])

def threshold_calibration_rows(splits, test_day):
    # Benign rows for choosing the autoencoder threshold: Monday hold-out + benign flows of the days
    # from Tuesday up to (not including) test_day. Never the training rows, never the test day.
    before = DAY_ORDER[1:DAY_ORDER.index(test_day)]
    mask = (splits["ae_role"] == "ae_val") | (splits["day"].isin(before) & (splits["is_attack"] == 0))
    return splits.index[mask].to_numpy()

for d in ["Tuesday", "Wednesday", "Thursday", "Friday"]:
    print(f"Threshold calibration flows for test day {d}: {len(threshold_calibration_rows(splits, d)):,}")
print(f"Autoencoder training flows: {len(ae_train):,} | early-stopping flows: {len(ae_val):,}")

**What this shows** (`03_split_counts_autoencoder.csv`): the autoencoder will be trained on 394,060 benign Monday flows, with 98,516 held back for early stopping. No attack flow is in either set (checked in Section 9).

## 7. Random reference split (identical to CW1)
**What:** repeats CW1's 70/15/15 split, stratified by label with the same seed, and checks that the count of every label in each part matches CW1's saved `02_split_counts.csv`.

**Why:** this split is used for two things in CW2: the reference result for attack families the model has already seen, and the held-out attack family experiment (E3), which repeats CW1's experiment with the autoencoder added. Matching CW1 exactly means the CW2 numbers can be compared directly with the CW1 report. Its known weakness, related flows from one session on both sides, is the reason it is a reference and not the main result.

In [ ]:
r_train, r_temp = train_test_split(idx_all, test_size=0.30, stratify=df["Label"], random_state=RANDOM_STATE)
r_val, r_test = train_test_split(r_temp, test_size=0.50, stratify=df["Label"].iloc[r_temp],
                                 random_state=RANDOM_STATE)
role = np.full(len(df), "unused", dtype=object)
role[r_train], role[r_val], role[r_test] = "train", "val", "test"
splits["random"] = pd.Categorical(role, categories=SPLIT_CATS)

cw1_counts = pd.read_csv(os.path.join(CW1_OUT, "02_split_counts.csv"), index_col=0).drop(index="TOTAL")
cw2_counts = (pd.crosstab(splits["Label"], splits["random"])
                .rename(columns={"val": "validation"})[["train", "validation", "test"]])
matches = cw2_counts.reindex(cw1_counts.index).astype(int).equals(cw1_counts[["train", "validation", "test"]].astype(int))
print("Random split label counts identical to CW1:", matches)
assert matches, "Random split differs from CW1; check the data and seed"

**What this shows:** the run printed `True`, so the count of every label in training, validation and test is identical to CW1 (1,747,580 / 374,482 / 374,482). CW2 results on this split can be compared directly with the CW1 report.

## 8. Split counts per label (evidence for the documentation)
**What:** saves, for every experiment, how many flows of each label fall in training, validation and test.

**Why:** the brief asks for class counts and the training and testing approach to be documented. These tables also show at a glance which labels are absent from training in each experiment.

In [ ]:
count_tables = {}
for col in ["E1", "E2_Wednesday", "E2_Thursday", "random"]:
    t = pd.crosstab(splits["Label"], splits[col]).reindex(columns=SPLIT_CATS, fill_value=0)
    t.loc["TOTAL"] = t.sum()
    t.to_csv(f"{OUT_DIR}/03_split_counts_{col}.csv")
    count_tables[col] = t

ae_counts = splits["ae_role"].value_counts().reindex(["ae_train", "ae_val"])
ae_counts.to_csv(f"{OUT_DIR}/03_split_counts_autoencoder.csv")

print("E1 (train Monday to Thursday, test Friday):")
display(count_tables["E1"])

## 9. Integrity checks
**What:**
1. In each time-ordered split, no test-day flow is in training or validation, and no training day appears in the test set.
2. How many test flows have exactly the same 62 feature values as some training flow, for each split.

**Why:** check 1 proves the split does what it claims. Check 2 measures overlap that day boundaries alone cannot remove. Notebook 01 dropped exact duplicate rows, but two flows can still share all 62 features if, for example, their destination ports differ. A high share of identical attack flows across training and test would mean a model can score well partly by recognising flows it has already seen. Comparing the random split with E1 shows how much the time-ordered split reduces that problem. This only catches exact copies, not near-duplicates, which remains a stated limitation.

In [ ]:
for col, test_day in [("E1", "Friday"), ("E2_Wednesday", "Wednesday"), ("E2_Thursday", "Thursday")]:
    s = splits[col]
    assert set(splits.loc[s == "test", "day"].astype(str)) == {test_day}
    train_days = set(splits.loc[s.isin(["train", "val"]), "day"].astype(str))
    assert all(DAY_ORDER.index(d) < DAY_ORDER.index(test_day) for d in train_days), f"{col}: later day in training"
assert set(splits.loc[splits["ae_role"] != "not_used_for_training", "day"].astype(str)) == {"Monday"}
assert splits.loc[splits["ae_role"] != "not_used_for_training", "is_attack"].sum() == 0
print("Time-order checks passed: every training flow comes from a day before its test day.")

row_hash = pd.util.hash_pandas_object(df[feature_cols], index=False).to_numpy()
rows = []
for col in ["E1", "E2_Wednesday", "E2_Thursday", "random"]:
    s = splits[col].to_numpy()
    seen = np.isin(row_hash[s == "test"], row_hash[(s == "train") | (s == "val")])
    test_attack = splits["is_attack"].to_numpy()[s == "test"] == 1
    rows.append({
        "split": col,
        "test_flows": int((s == "test").sum()),
        "identical_to_a_training_flow": int(seen.sum()),
        "share_pct": round(100 * seen.mean(), 3),
        "attack_test_flows": int(test_attack.sum()),
        "attack_flows_identical_to_training": int((seen & test_attack).sum()),
        "attack_share_pct": round(100 * (seen & test_attack).sum() / max(test_attack.sum(), 1), 3),
    })
overlap = pd.DataFrame(rows)
overlap.to_csv(f"{OUT_DIR}/03_train_test_identical_flows.csv", index=False)
display(overlap)

**What this shows** (`03_train_test_identical_flows.csv`):

| Split | Attack test flows | Identical to a training flow | Share |
|---|---|---|---|
| Random (CW1) | 63,751 | 13,375 | 20.98% |
| E1 (Friday) | 220,089 | 34,182 | 15.53% |
| E2 Wednesday | 193,590 | 5 | 0.003% |
| E2 Thursday | 2,179 | 0 | 0% |

- In the random split, about 1 in 5 attack test flows has exactly the same 62 feature values as a flow in training. This supports the CW1 caveat that random-split scores are probably optimistic.
- The E1 figure needs care. Friday's attack families never appear in training, yet 34,182 Friday attack flows are identical to some Monday to Thursday flow. Those matches must be to flows with a different label, for example benign traffic. Section 9b checks which.
- On Wednesday and Thursday the overlap is close to zero, so those tests are almost free of exact copies.

### 9b. What are the identical attack flows identical to?
**What:** for every attack flow in the test set that has an exact copy in training, checks whether the copy is labelled benign, the same attack label, or a different attack label.

**Why:** the answer changes the interpretation. A copy with the same label means the model may simply recognise a flow it has already seen, which inflates scores (leakage). A copy labelled benign means the attack flow is indistinguishable from normal traffic using these 62 features, so no flow-by-flow model can detect it. That would set a ceiling on detection for that attack type and would be a key limitation for CW2's evaluation, especially for PortScan, whose flows are often only one or two packets.

In [ ]:
train_hash_labels = None
rows = []
for col in ["random", "E1"]:
    s = splits[col].to_numpy()
    in_train = (s == "train") | (s == "val")
    test_attack = (s == "test") & (splits["is_attack"].to_numpy() == 1)

    train_pairs = pd.DataFrame({"h": row_hash[in_train], "label": labels[in_train]}).drop_duplicates()
    benign_hashes = train_pairs.loc[train_pairs["label"] == "BENIGN", "h"].to_numpy()
    attack_pairs = train_pairs[train_pairs["label"] != "BENIGN"]

    t = pd.DataFrame({"Label": labels[test_attack], "h": row_hash[test_attack]})
    t["any_copy"] = np.isin(t["h"], train_pairs["h"].to_numpy())
    t["copy_is_benign"] = np.isin(t["h"], benign_hashes)
    same = pd.MultiIndex.from_frame(attack_pairs[["h", "label"]])
    t["copy_has_same_label"] = pd.MultiIndex.from_arrays([t["h"], t["Label"]]).isin(same)
    t["copy_is_other_attack"] = np.isin(t["h"], attack_pairs["h"].to_numpy()) & ~t["copy_has_same_label"]

    b = t.groupby("Label").agg(attack_test_flows=("h", "size"), identical_to_any=("any_copy", "sum"),
                               copy_is_benign=("copy_is_benign", "sum"),
                               copy_has_same_label=("copy_has_same_label", "sum"),
                               copy_is_other_attack=("copy_is_other_attack", "sum"))
    b.insert(0, "split", col)
    rows.append(b.reset_index())

identical_breakdown = pd.concat(rows, ignore_index=True)
identical_breakdown.to_csv(f"{OUT_DIR}/03_identical_flows_breakdown.csv", index=False)
display(identical_breakdown[identical_breakdown["identical_to_any"] > 0])
print("Note: one test flow can match several training flows, so the three 'copy' columns can overlap.")

## 10. Chart: which days each experiment uses
**What:** one row per experiment, one column per day, coloured by what the day is used for.

**Why:** the testing design is the hardest part of CW2 to explain in words. This chart is intended for the presentation's testing methodology slide and the documentation.

In [ ]:
colours = {"Train and validation": "#4C72B0", "Test": "#DD8452", "Threshold calibration (benign only)": "#8C8C8C",
           "Autoencoder training (benign only, 20% held out)": "#55A868", "Not used": "#FFFFFF"}
plan = {
    "E1 (main)":       ["Train and validation"] * 4 + ["Test"],
    "E2: Wednesday":   ["Train and validation"] * 2 + ["Test", "Not used", "Not used"],
    "E2: Thursday":    ["Train and validation"] * 3 + ["Test", "Not used"],
    "Autoencoder, E1": ["Autoencoder training (benign only, 20% held out)"] + ["Threshold calibration (benign only)"] * 3 + ["Test"],
}
fig, ax = plt.subplots(figsize=(10, 3.2))
for r, (name, cells) in enumerate(plan.items()):
    for c, use in enumerate(cells):
        ax.add_patch(plt.Rectangle((c, r), 0.95, 0.85, facecolor=colours[use], edgecolor="#333333"))
ax.set_xlim(0, 5); ax.set_ylim(len(plan), -0.15)
ax.set_xticks(np.arange(5) + 0.475); ax.set_xticklabels(DAY_ORDER)
ax.set_yticks(np.arange(len(plan)) + 0.425); ax.set_yticklabels(list(plan))
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.legend(handles=[Patch(facecolor=v, edgecolor="#333333", label=k) for k, v in colours.items()],
          loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=8)
ax.set_title("CW2 time-ordered testing design (CICIDS2017 capture days)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_split_design.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Save the splits and the run configuration
**What:** saves the split assignment of every flow, then reloads it and checks it matches. Also saves the seeds, fractions, library versions and data fingerprint.

**Why:** Notebooks 04 to 06 load these splits instead of rebuilding them, so every model is trained and tested on exactly the same flows. The configuration file is the reproducibility evidence the brief asks for.

In [ ]:
splits.to_parquet(SPLIT_FILE, index=False)
check = pd.read_parquet(SPLIT_FILE)
for col in ["E1", "E2_Wednesday", "E2_Thursday", "random", "ae_role"]:
    assert (check[col].astype(str).to_numpy() == splits[col].astype(str).to_numpy()).all(), col
print("Saved and verified:", SPLIT_FILE, f"({os.path.getsize(SPLIT_FILE) / 1e6:.1f} MB)")

config = {
    "notebook": "03_data_and_splits",
    "rows": int(len(df)),
    "n_features": len(feature_cols),
    "data_fingerprint": FINGERPRINT,
    "random_state": RANDOM_STATE,
    "val_frac_chronological": VAL_FRAC,
    "ae_val_frac": AE_VAL_FRAC,
    "day_order": DAY_ORDER,
    "experiments": {
        "E1": "train/val Monday-Thursday (val 15% stratified by Label), test Friday",
        "E2_Wednesday": "train/val Monday-Tuesday, test Wednesday",
        "E2_Thursday": "train/val Monday-Wednesday, test Thursday",
        "random": "CW1 70/15/15 split stratified by Label, seed 42 (verified identical to CW1)",
        "autoencoder": "train on 80% of benign Monday, early stopping on 20%; threshold on benign "
                       "flows from the Monday hold-out and Tuesday up to the day before the test day",
    },
    "split_counts": {col: splits[col].value_counts().reindex(SPLIT_CATS).astype(int).to_dict()
                     for col in ["E1", "E2_Wednesday", "E2_Thursday", "random"]},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "sklearn": sklearn.__version__,
                 "matplotlib": matplotlib.__version__},
}
with open(f"{OUT_DIR}/03_run_config.json", "w") as fh:
    json.dump(config, fh, indent=2)
print(json.dumps(config["split_counts"], indent=2))

## What to check after running

The "What this shows" notes above come from the run of 14 Nov 2026. After re-running, please push `cw2/outputs/03_identical_flows_breakdown.csv` so the Section 9b note can be written from the real result.

1. **Section 4:** Monday has no attacks, every attack label appears on one day, and the list of Friday families never seen Monday to Thursday.
2. **Section 7:** "Random split label counts identical to CW1: True".
3. **Section 9:** the share of test flows identical to a training flow, especially for attack flows, in the random split compared with E1.
4. **Section 10:** the design chart in `cw2/outputs/03_split_design.png`.

**Questions to be ready for**
- *Why not a random split?* Related flows from one attack session can be on both sides, which inflates scores. Section 9 measures part of that effect.
- *Why is Friday the main test day?* It is the last day, so training on earlier days keeps time order, and it contains PortScan, DDoS and Botnet, which match TeleComNet's scanning and denial-of-service concerns.
- *Why can Tuesday not be a supervised test day?* Monday has no attacks, so there is nothing for a supervised model to learn from.
- *Why train the autoencoder only on Monday?* It is the only day with no attacks, so the model learns normal traffic only. Monday's normal traffic may differ from Friday's, which could raise false alarms; Notebook 05 measures that.